In [1]:
import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

# 1. Запускаем сам сервер Phoenix
session = px.launch_app()

# 2. Регистрируем провайдер трассировки (OpenTelemetry)
# Он будет перехватывать данные и отправлять их в локальный Phoenix
tracer_provider = register()

# 3. Включаем "прослушку" именно для LangChain
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

print(f"Phoenix готов! Дашборд тут: {session.url}")

/Users/artemzmailov/Desktop/kitoboy-PII/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/artemzmailov/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/Users/artemzmailov/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)
E0425 23:22:20.239218 4337980 add_port.cc:83] Failed to add port to server: No address added out of total 1 resolved for '[::]:4317'
ERROR:    Traceback (most recent call last):
  File "/Users/artemzmailov/Desktop/kitoboy-PII/.venv/lib/python3.11/site-packages/starlette/routing.

🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix готов! Дашборд тут: http://localhost:6006/


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Теперь к ним можно обращаться через стандартный модуль os
api_key = os.getenv("MISTRAL_API_KEY")
print(api_key) 

Lld8KeTo2clg7lurqzk9G0XZlCuGBdhx


In [3]:
import yaml
import re
from typing import List, Set, TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

# Модель для structured_output (как ты и писал)
class Entities(BaseModel):
    items: List[str] = Field(description="Список сгенерированных сущностей")

# Состояние нашего подграфа
class SubAgentState(TypedDict):
    entity_key: str          # Ключ из YAML (например, 'PASSPORT_RF')
    target_count: int        # Сколько всего нужно уникальных штук
    unique_samples: Set[str] # Наше множество (авто-дедупликация)
    last_batch: List[str]    # Последний выхлоп модели (для логов/проверки)
    iterations: int             # Счетчик итераций

In [4]:
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_mistralai import ChatMistralAI

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.3,   
    check_every_n_seconds=0.1,
    max_bucket_size=1,
)

In [5]:
from langchain_mistralai import ChatMistralAI

# Инициализация модели
llm = ChatMistralAI(model="mistral-small-2506", #mistral-small-2506
                    temperature=0.7,
                    timeout = 60,
                    rate_limiter=rate_limiter,
                    max_concurrent_requests=1,
                    max_retries=3)
structured_llm = llm.with_structured_output(Entities)

#llm = ChatOllama(model="qwen3.5:9b", reasoning = False, format="json", temperature=0.7)
# structured_llm = llm.with_structured_output(Entities)

# Загрузка промптов (предположим, файл лежит рядом)
with open('../../configs/prompts.yaml', 'r', encoding='utf-8') as f:
    PROMPTS_CONFIG = yaml.safe_load(f)

def generate_batch_node(state: SubAgentState):
    entity_key = state['entity_key']
    config = PROMPTS_CONFIG.get('ENTITIES', {}).get(entity_key)
    
    if config is None:
        raise ValueError(f"Сущность {entity_key} не найдена в секции ENTITIES в prompts.yaml")
    
    # Считаем, сколько еще не хватает до цели
    needed = state['target_count'] - len(state['unique_samples'])
    batch_size = min(needed, 10) # Генерим не больше 10 за раз для качества
    
    # Берем системный промпт и подставляем N
    sys_prompt = config['system_prompt'].format(N=batch_size)
    
    # Формируем подсказку, чтобы не повторяться (берем последние 5 примеров)
    history = list(state['unique_samples'])[-5:]
    user_content = f"Сгенерируй {batch_size} новых примеров."
    if history:
        user_content += f" Не повторяй эти форматы: {history}. Попробуй другие форматы записи, разделители или цифры."
    
    messages = [
        SystemMessage(content=sys_prompt),
        HumanMessage(content=user_content)
    ]
    
    # Вызов модели
    response = structured_llm.invoke(messages)
    
    return {
        "last_batch": response.items,
        "iterations": state['iterations'] + 1
    }

def validate_and_add_node(state: SubAgentState):
    # Просто добавляем всё, что выдала модель, в set()
    # Так как мы отказались от регулярок, тут только дедупликация
    new_items = state['last_batch']
    updated_samples = state['unique_samples'].copy()
    
    for item in new_items:
        if item and len(item) > 5: # Базовый фильтр от пустых строк
            updated_samples.add(item)
            
    return {"unique_samples": updated_samples}

def should_continue(state: SubAgentState):
    # Если набрали нужное количество или превысили лимит попыток (напр. 20)
    if len(state['unique_samples']) >= state['target_count'] or state['iterations'] > 20: 
        return "end"
    return "continue"

In [6]:
# Собираем граф
workflow = StateGraph(SubAgentState)

# Добавляем узлы
workflow.add_node("generator", generate_batch_node)
workflow.add_node("validator", validate_and_add_node)

# Устанавливаем точку входа
workflow.set_entry_point("generator")

# Связываем узлы
workflow.add_edge("generator", "validator")

# Добавляем условный переход из валидатора
workflow.add_conditional_edges(
    "validator",
    should_continue,
    {
        "continue": "generator",
        "end": END
    }
)

# Компилируем
sub_agent_app = workflow.compile()

In [7]:
input_state = {
    'entity_key': 'VK',       
    'target_count': 100,
    'unique_samples': set(),
    'iterations': 0            
}
result = sub_agent_app.invoke(input_state,output_keys = ['entity_key', 'unique_samples'])

In [8]:
result

{'entity_key': 'VK',
 'unique_samples': {'@ivan_ivanov_msk',
  '@maria_ivanova',
  '@maria_ivanova_2001',
  '@maria_ivanova_msk',
  '@maria_ivanova_nsk',
  '@maria_kuznetsova',
  '@marina_kuznetsova',
  '@petr_ivanov_msk',
  '[id12345678|Антон]',
  '[id12345678|Ольга Петрова]',
  '[id456789123|Иван]',
  '[id555666777|Светлана]',
  '[id5566778899|Мария]',
  '[id777888999|Дмитрий]',
  '[id777888|Мария]',
  '[id87654321|Алексей Петров]',
  '[id9988776655|Иван]',
  '[id99887766|Сергей]',
  'https://vk.com/alex_ivanov_msk',
  'https://vk.com/alexey_petrov_msk',
  'https://vk.com/club12345678',
  'https://vk.com/id98765432',
  'https://vk.me/id444555666',
  'https://vkontakte.ru/nikita_petrov_ekb',
  'm.vk.com/anna_kovalenko_2004',
  'm.vk.com/ivan_ivanov_spb',
  'm.vk.com/my_friend_1990',
  'm.vk.com/my_page_2026',
  'm.vk.com/olga_ivanova_spb',
  'm.vk.com/olga_petrova',
  'm.vk.com/olga_smirnova',
  'm.vk.com/olga_smirnova_66',
  'm.vk.com/olga_smirnova_nsk',
  'vk.com/[id12345678|Мария И

In [39]:
import os
import json

FOLDER_PATH = '../outputs/'

entity_name = result['entity_key']
samples = result['unique_samples']

with open(FOLDER_PATH + f'{entity_name}.json', 'r') as f:
    file = json.load(f)
    print('Current len:', len(file))
new = set(file)
new.update(samples)
print('New len:', len(new))


Current len: 20
New len: 107


In [40]:
with open(FOLDER_PATH + f'{entity_name}.json', 'w') as f:
    json.dump(list(new), f, ensure_ascii = False, indent = 4)

In [31]:
# import os
# import json
# FOLDER_PATH = '../outputs/'
# categories = ['PASSPORT_RF', 'BANK_CARD', 'PHONE_NUMBER', 'EMAIL', 'TELEGRAM', 'VK']
# for cat in categories:
#     input_state = {
#     'entity_key': cat,       
#     'target_count': 10,
#     'unique_samples': set(),
#     'iterations': 0            
#     }
#     res = sub_agent_app.invoke(input_state,output_keys = ['unique_samples'])['unique_samples']
#     print(cat)
#     print(res)
#     with open(FOLDER_PATH + f'{cat}.json', 'w') as f:
#         json.dump(list(res), f, ensure_ascii = False, indent = 4)
#     break
        
    

PASSPORT_RF
{'паспорт сер. 4511 № 123456', '4501 112233', '4612 889900', '4511123456', '4002-987654', '4005 654321', '0118-554433', '7715 001234', '6010-776655', '2014 334455'}
